# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 clinical oncology dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record sets and their fields by @id
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    # recordSet may be a list or single object
    record_sets = metadata.recordSet if isinstance(metadata.recordSet, list) else [metadata.recordSet]
else:
    # Try to find record sets from the dataset using mlcroissant API (fallback if not in top-level metadata)
    record_sets = dataset.list_record_sets()
    # This returns a list of @id strings

print("Available Record Sets (referenced by @id):")
for rs_id in record_sets:
    print(f" - {rs_id}")

# Print available fields for each record set
for rs_id in record_sets:
    print(f"\nRecord Set @id: {rs_id}")
    field_ids = dataset.list_fields(record_set=rs_id)
    print("Fields (@id):")
    for f_id in field_ids:
        print(f"    - {f_id}")
    # Optionally show a sample record
    sample = next(dataset.records(record_set=rs_id), None)
    if sample:
        print("\nSample record:")
        print(sample)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for Record Set {record_set_id}:")
        print(df.columns.tolist())
        print(df.head())

# Choose the first available record set for further analysis
if len(dataframes):
    analysis_rs_id = list(dataframes.keys())[0]
    df = dataframes[analysis_rs_id]
    print(f"\nProceeding with Record Set: {analysis_rs_id}")
else:
    print("No record sets found for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a numeric field (@id) for demonstration
# We'll pick the first numeric-looking column (e.g., 'age' or 'Interval_between_cancers_days' if present)

numeric_candidates = [col for col in df.columns if ('age' in col.lower()) or ('interval' in col.lower()) or ('days' in col.lower())]
if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Selected numeric field: {numeric_field}")
else:
    numeric_field = df.select_dtypes(include='number').columns[0] if not df.select_dtypes(include='number').empty else df.columns[0]
    print(f"Fallback numeric field: {numeric_field}")

# Set a threshold for filtering
threshold = 50
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field (@id)
# Candidate group field: 'Sex', 'MSI_status', or 'Cancer_Type'
categorical_candidates = [col for col in df.columns if any(key in col.lower() for key in ['sex', 'msi', 'cancer', 'location', 'anatomical'])]
group_field = categorical_candidates[0] if categorical_candidates else df.columns[0]
if group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Plot histogram of numeric field
plt.figure(figsize=(6, 4))
df[numeric_field].hist(bins=15)
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.title(f'Distribution of {numeric_field}')
plt.show()

# Boxplot of numeric field grouped by categorical field
if group_field in df.columns:
    plt.figure(figsize=(8, 6))
    df.boxplot(column=numeric_field, by=group_field)
    plt.title(f'{numeric_field} by {group_field}')
    plt.suptitle('')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR^2 clinical oncology dataset using the `mlcroissant` library. We reviewed dataset metadata, extracted records using the schema's `@id` references, applied basic data processing, and visualized important clinical features. The dataset supports research into clinicopathological predictors and MSI-H phenotype distribution among cancer survivors.

You can extend this exploration with further statistical analysis, cross-record set joins, or machine learning tasks tailored to the clinical questions of interest.